# Coverage contribution to the MEG–iEEG discrepancy

Construct five MEG datasets, compare their total variance and spatial coverage, and fit **separate PCA models with up to 10 components** to each dataset and the pooled iEEG reference.

| Setup | Construction |
|---|---|
| `full_average` | All MEG sources, averaged across all MEG participants |
| `full_concatenated` | All sources from every MEG participant, concatenated as features |
| `coverage_average` | For each pooled iEEG electrode, find the nearest MEG source in every participant, then average participants |
| `paired_coverage` | Pair each iEEG participant with one MEG participant, select sources near that participant's electrodes, then concatenate |
| `random_control` | The **same participant pairing** and per-participant electrode counts, but spatially random source selection |

The group-coverage definition follows `match_meg(..., average_subject=True)`, not the older radius-mask implementation. One output feature is retained per iEEG electrode, including repeated nearest-source matches. By default, the random control preserves those repeated-source multiplicities too, so differences are not simply due to feature duplication.

The original `match_meg` used the paired participant's coordinates but read `meg_data_source[i]`; the helper here uses the paired participant for **both**. It also samples pairings from **all** available MEG participants rather than only the first `n_iEEG` participants.

**Scope:** descriptive coverage/aggregation ablation using trial-averaged data. This notebook does not establish held-out generalisation or memory specificity, and electrode/time samples are not independent population replicates.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Run from iEEGvsMEG, or from its parent repository directory.
ROOT = Path.cwd()
if not (ROOT / 'utils_updated.py').exists() and (ROOT / 'iEEGvsMEG' / 'utils_updated.py').exists():
    ROOT = ROOT / 'iEEGvsMEG'
if not (ROOT / 'utils_updated.py').exists():
    raise FileNotFoundError('Set ROOT to the directory containing coverage_matching.ipynb and utils_updated.py.')
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT.parent))  # optional original src.setting/GetInfo
from utils_updated import (
    load_project_data, construct_five_datasets, make_ieeg_dataset,
    variance_summary, fit_block_pca, compare_to_ieeg, plot_coverage,
    plot_pca, plot_explained_variance, plot_correlations,
    plot_time_overlays, synthetic_inputs,
)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})

## Configuration and input contract

MEG files: `MEG/dataMEG/<subject>_source.p` (condition × source × time) and `<subject>_pos.csv` (ordered source coordinates). iEEG files: `ieeg_shortWOBS_fs250/<subject>_epochs.p` (trial × channel × time) and `<subject>_info.json` with `event_id` and `time_epoch`. Trial averages are calculated per condition without the old fixed 24-trial limit.

Supply electrode metadata through the original `src.setting.GetInfo` or a CSV containing `subject,channel_index,x,y,z`, where `channel_index` is **zero-based in the saved epoch channel order**. CSV rows are re-ordered explicitly by these keys. Optional columns such as `channel` and `region` are preserved. For GetInfo, verify its within-participant electrode ordering against the saved epochs.

Coordinates must be in one explicit unit per input, in the same MNI space. The old element-wise unit-repair heuristic is deliberately not applied. Inspect the upstream metadata if the unit/range check fails.

Set either `MEG_TIMES_FILE` (one-dimensional `.npy` or headerless `.csv`, in seconds) or the **verified** `MEG_TMIN`. The original extra final MEG sample is trimmed only when the remaining times match iEEG. The time origin is never guessed from iEEG.

Default preprocessing reproduces the example pipeline: MEG channel z-score across condition and time before participant aggregation, iEEG multiplied by 1000 to reverse the stated upstream scaling, and conditions averaged before PCA. Raw absolute variance across modalities is therefore not a directly comparable physical quantity. For a symmetric preprocessing sensitivity analysis, use channel z-scoring in both modalities. `CONDITION_MODE='stack'` preserves separate conditions as consecutive observations; use the same mode for every dataset.

In [ ]:
# Real data are the default. Synthetic mode is only a runnable demonstration.
SYNTHETIC_DEMO = False
MEG_DIR = ROOT / 'MEG' / 'dataMEG'
IEEG_DIR = ROOT / 'ieeg_shortWOBS_fs250'
METADATA_CSV = None  # e.g. ROOT / 'ieeg_electrodes.csv'; otherwise original GetInfo
PROJECT_PATH = None  # override GetInfo's original PROJECT_PATH if necessary
MEG_COORDINATE_UNIT = 'm'
IEEG_COORDINATE_UNIT = 'mm'  # explicitly verify for your metadata export
MEG_TIMES_FILE = None  # e.g. MEG_DIR / 'time_meg.npy'
MEG_TMIN = None        # set from the original acquisition metadata if no time file
SFREQ = 250.
CONDITIONS = (1, 2)    # saved MEG condition axis must have this same order
CONDITION_MODE = 'average'  # 'average' (legacy) or 'stack'
MEG_SCALING = 'channel_zscore'
IEEG_SCALING = 'none'
IEEG_MULTIPLIER = 1000.
SEED = 2026
PAIRING = None  # optional fixed dict: {'iEEG_subject': 'MEG_subject', ...}
RANDOM_PRESERVE_DUPLICATES = True
N_COMPONENTS = 10
FEATURE_CHUNK = 1024
MAX_GRAM_GIB = 2.  # approximate budget for three observation Gram arrays
MAX_COVERAGE_POINTS = 20000  # skip full concatenation coverage when oversized
N_REPEATS = 0  # optional robustness runs; try 20 after the main analysis
OUT_DIR = ROOT / ('out/coverage_matching_demo' if SYNTHETIC_DEMO else 'out/coverage_matching')

In [ ]:
if SYNTHETIC_DEMO:
    print('SYNTHETIC DEMONSTRATION — these are not study results.')
    inputs = synthetic_inputs(SEED)
else:
    inputs = load_project_data(
        MEG_DIR, IEEG_DIR, metadata_csv=METADATA_CSV, project_path=PROJECT_PATH,
        meg_coordinate_unit=MEG_COORDINATE_UNIT,
        ieeg_coordinate_unit=IEEG_COORDINATE_UNIT,
        meg_scaling=MEG_SCALING, ieeg_scaling=IEEG_SCALING,
        ieeg_multiplier=IEEG_MULTIPLIER, meg_times_file=MEG_TIMES_FILE,
        meg_tmin=MEG_TMIN, sfreq=SFREQ, conditions=CONDITIONS,
    )
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"MEG participants: {len(inputs['meg_subjects'])}; iEEG participants: {len(np.unique(inputs['electrode_subjects']))}")
print(f"MEG per participant: {inputs['meg'][0].shape}; pooled iEEG: {inputs['ieeg'].shape}")
print(f"Time: {inputs['times'][0]:.4f} to {inputs['times'][-1]:.4f} s, {len(inputs['times'])} samples")
display(inputs['trial_counts'])
display(inputs['electrode_metadata'].head())
print('MNI coordinate ranges, mm:')
display(inputs['electrode_metadata'][['x', 'y', 'z']].agg(['min', 'max']))

## Construct the five datasets and audit matching

Full averaging requires a common ordered source grid. The helper fails explicitly if grids differ; morph/register them upstream rather than averaging unrelated source indices. Subject pairing is one-to-one without replacement and requires at least as many MEG as iEEG participants. All participants contribute to the first three setups; only paired participants contribute to the last two. This participant-count difference remains a potential contributor when contrasting those groups.

The pairing, random source choices, distances and source duplications are saved. Inspect long-distance matches before interpreting the analysis; this notebook retains them to keep electrode counts constant.

In [ ]:
meg_datasets, matching_audit, pairing = construct_five_datasets(
    inputs['meg'], inputs['meg_positions'], inputs['meg_subjects'],
    inputs['electrode_positions'], inputs['electrode_subjects'],
    seed=SEED, pairing=PAIRING, condition_mode=CONDITION_MODE,
    random_preserve_duplicates=RANDOM_PRESERVE_DUPLICATES,
)
ieeg_dataset = make_ieeg_dataset(inputs['ieeg'], inputs['electrode_metadata'], CONDITION_MODE)
all_datasets = {'iEEG': ieeg_dataset, **meg_datasets}
assert all(ds.n_observations == ieeg_dataset.n_observations for ds in meg_datasets.values())
matching_audit.to_csv(OUT_DIR / 'matching_audit.csv', index=False)
inputs['electrode_metadata'].to_csv(OUT_DIR / 'ieeg_electrode_metadata.csv', index=False)
inputs['trial_counts'].to_csv(OUT_DIR / 'trial_counts.csv', index=False)
pairing_summary = matching_audit.groupby(['ieeg_subject', 'meg_subject']).agg(
    n_electrodes=('electrode_index', 'size'),
    n_unique_matched_sources=('source_index', 'nunique'),
    n_unique_random_sources=('random_source_index', 'nunique'),
    median_distance_mm=('distance_mm', 'median'),
    max_distance_mm=('distance_mm', 'max'),
    median_random_distance_mm=('random_distance_mm', 'median'),
)
display(pairing_summary)
pairing_summary.to_csv(OUT_DIR / 'pairing_summary.csv')
for name, ds in meg_datasets.items():
    ds.metadata.to_csv(OUT_DIR / f'{name}_feature_metadata.csv', index=False)
    np.save(OUT_DIR / f'{name}_electrode_to_feature.npy', ds.electrode_to_feature)

## Dataset variance and coverage — before PCA

**Total variance** is the sum of channel-wise sample variances across PCA observations (trace of the centred sample covariance, `ddof=1`). It depends on channel count. **Mean feature variance** divides this by the number of features. These are measured *after the configured input preprocessing and aggregation*, without additional PCA whitening or scaling.

Duplicated nearest-source features count repeatedly, as they do in the PCA objective. Coverage displays actual selected source coordinates and iEEG contacts in MNI millimetres. Co-located concatenated features overlap in these plots; feature counts and unique-position counts distinguish multiplicity from spatial extent.

In [ ]:
variance_table = variance_summary(all_datasets)
display(variance_table)
variance_table.to_csv(OUT_DIR / 'variance_summary.csv')
fig, axes = plt.subplots(1, 2, figsize=(14, 4), constrained_layout=True)
for ax, column in zip(axes, ['total_variance', 'mean_feature_variance']):
    variance_table[column].plot.bar(ax=ax)
    ax.set(title=column.replace('_', ' ').title(), ylabel='Squared processed-signal units')
    ax.tick_params(axis='x', labelrotation=35)
fig.savefig(OUT_DIR / 'variance_summary.png', dpi=160)
plt.show()
plt.close(fig)

In [ ]:
coverage_figures = plot_coverage(meg_datasets, inputs['electrode_positions'], MAX_COVERAGE_POINTS)
for name, fig in coverage_figures:
    fig.savefig(OUT_DIR / f'coverage_{name}.png', dpi=160)
    display(fig)
    plt.close(fig)
del coverage_figures

## Separate PCA models and the first 10 temporal components

PCA uses features as columns and time (or condition × time) as observations. The exact centred PCA is calculated from the observation Gram matrix in feature chunks, so the full-concatenation dataset does not require a huge dense concatenated matrix or channel covariance matrix. Runtime still increases with the number of participants, sources and observations.

No whitening or post-aggregation channel standardisation is applied. The denominator for explained-variance ratios includes **all** variance, not just the first ten PCs. Numerical rank can limit the number of nonzero components; this is reported rather than plotting artificial zero components.

In [ ]:
pca_results = {}
for name, ds in all_datasets.items():
    print(f'PCA: {name} ({ds.n_observations:,} observations × {ds.n_features:,} features)', flush=True)
    result = fit_block_pca(ds, N_COMPONENTS, FEATURE_CHUNK, MAX_GRAM_GIB)
    pca_results[name] = result
    np.testing.assert_allclose(result.total_variance, variance_table.loc[name, 'total_variance'], rtol=1e-6)
    np.savez_compressed(
        OUT_DIR / f'{name}_pca.npz', scores=result.scores, weights=result.weights,
        explained_variance=result.explained_variance,
        explained_variance_ratio=result.explained_variance_ratio,
        total_variance=result.total_variance,
    )

In [ ]:
for name, result in pca_results.items():
    fig = plot_pca(result, name, inputs['times'], CONDITION_MODE, CONDITIONS)
    fig.savefig(OUT_DIR / f'pca_timecourses_{name}.png', dpi=160)
    display(fig)
    plt.close(fig)
fig = plot_explained_variance(pca_results)
fig.savefig(OUT_DIR / 'pca_explained_variance.png', dpi=160)
plt.show()
plt.close(fig)
explained_table = pd.concat([
    pd.DataFrame({'dataset': name, 'pc': np.arange(1, len(r.explained_variance) + 1),
                  'explained_variance': r.explained_variance,
                  'explained_variance_ratio': r.explained_variance_ratio,
                  'cumulative_ratio': np.cumsum(r.explained_variance_ratio)})
    for name, r in pca_results.items()
], ignore_index=True)
explained_table.to_csv(OUT_DIR / 'pca_explained_variance.csv', index=False)
display(explained_table.pivot(index='pc', columns='dataset', values='explained_variance_ratio'))

## Correlation with iEEG PCA: temporal scores and weights

Every heatmap contains **all PC × PC correlations** (up to 10 × 10), so a change in component ordering is visible. PCA signs are arbitrary; interpret correlation magnitude alongside signed values. Rank-wise overlays below align signs using temporal correlation only, and preserve PC order.

Weight correlations require the same feature identities. We use:

- Full average: weights at the nearest source for each iEEG electrode.
- Full concatenation: weights at the nearest source **in the fixed paired MEG participant** for each iEEG electrode.
- Group coverage and paired coverage: the stored electrode-order feature slots.
- Random control: the randomly assigned feature in each electrode slot. This is a **non-anatomical control**, not a claim about matched locations.

Thus full-source PCA is still fit using **all sources**; only its weights are sampled for comparison. Duplicated correspondences are retained and may affect correlations. The selected full-concatenation weights describe the paired participant blocks, not a unique whole-group spatial map.

A descriptive Hungarian assignment maximises absolute temporal correlation with one-to-one PC matching. Its sign is also applied to the paired weight correlation, rather than independently making weight correlations positive. This is an in-sample summary, not held-out validation or a significance test. No time-point/electrode-wise p-values are computed.

In [ ]:
reference = pca_results['iEEG']
comparisons = {name: compare_to_ieeg(ds, pca_results[name], reference)
               for name, ds in meg_datasets.items()}
assigned_table = pd.concat([c['assignment'] for c in comparisons.values()], ignore_index=True)
assigned_table.to_csv(OUT_DIR / 'time_selected_component_assignment.csv', index=False)
rank_rows = []
for name, comparison in comparisons.items():
    for kind in ['time', 'weights']:
        matrix = comparison[kind]
        pd.DataFrame(matrix, index=[f'MEG_PC{i+1}' for i in range(matrix.shape[0])],
                     columns=[f'iEEG_PC{i+1}' for i in range(matrix.shape[1])]).to_csv(OUT_DIR / f'{name}_{kind}_correlations.csv')
    for pc in range(min(comparison['time'].shape)):
        tr, wr = comparison['time'][pc, pc], comparison['weights'][pc, pc]
        rank_rows.append({'dataset': name, 'pc': pc + 1, 'time_r': tr,
                          'abs_time_r': abs(tr), 'weight_r': wr,
                          'time_aligned_weight_r': wr * (-1 if tr < 0 else 1)})
rank_table = pd.DataFrame(rank_rows)
rank_table.to_csv(OUT_DIR / 'rankwise_correlations.csv', index=False)
display(rank_table)
print('Descriptive one-to-one PC assignment selected by temporal correlation:')
display(assigned_table)
fig = plot_correlations(comparisons)
fig.savefig(OUT_DIR / 'ieeg_pca_correlations.png', dpi=160)
plt.show()
plt.close(fig)

In [ ]:
fig = plot_time_overlays({name: pca_results[name] for name in meg_datasets},
                         reference, inputs['times'], CONDITION_MODE)
fig.savefig(OUT_DIR / 'rankwise_timecourse_overlays.png', dpi=160)
plt.show()
plt.close(fig)

## Optional pairing/source-selection robustness

Set `N_REPEATS > 0` to repeat **paired coverage versus its random control**. When `PAIRING=None`, participants are re-paired on each run; an explicit `PAIRING` holds participants fixed and varies random source selection only. Both conditions use the same pairing within each run. The same iEEG reference is used throughout.

These are sensitivity distributions, not independent subject replicates or a permutation significance test. The summary is the mean absolute time correlation over the available first three rank-matched PCs; full per-PC assignments are also saved. This section intentionally does not repeat the expensive full-concatenation PCA.

In [ ]:
robustness_rows, robustness_assignments = [], []
for run in range(N_REPEATS):
    run_seed = SEED + 1 + run
    run_sets, run_audit, run_pairing = construct_five_datasets(
        inputs['meg'], inputs['meg_positions'], inputs['meg_subjects'],
        inputs['electrode_positions'], inputs['electrode_subjects'],
        seed=run_seed, pairing=PAIRING, condition_mode=CONDITION_MODE,
        random_preserve_duplicates=RANDOM_PRESERVE_DUPLICATES,
    )
    run_audit.to_csv(OUT_DIR / f'robustness_matching_{run:03d}.csv', index=False)
    for name in ['paired_coverage', 'random_control']:
        r = fit_block_pca(run_sets[name], N_COMPONENTS, FEATURE_CHUNK, MAX_GRAM_GIB)
        c = compare_to_ieeg(run_sets[name], r, reference)
        k = min(3, *c['time'].shape)
        robustness_rows.append({'run': run, 'seed': run_seed, 'dataset': name,
                                'mean_abs_time_r_first3_rankwise': np.abs(np.diag(c['time'])[:k]).mean()})
        robustness_assignments.append(c['assignment'].assign(run=run, seed=run_seed))
    del run_sets
    print(f'Robustness run {run+1}/{N_REPEATS}', flush=True)
if robustness_rows:
    robustness = pd.DataFrame(robustness_rows)
    robustness.to_csv(OUT_DIR / 'robustness_summary.csv', index=False)
    pd.concat(robustness_assignments, ignore_index=True).to_csv(OUT_DIR / 'robustness_assignments.csv', index=False)
    paired = robustness.pivot(index='run', columns='dataset', values='mean_abs_time_r_first3_rankwise')
    paired['coverage_minus_random'] = paired['paired_coverage'] - paired['random_control']
    display(paired.describe())
    fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
    for _, row in paired.iterrows():
        ax.plot([0, 1], row[['paired_coverage', 'random_control']], 'o-', alpha=.4)
    ax.set(xticks=[0, 1], xticklabels=['Paired coverage', 'Random source control'],
           ylabel='Mean |time r|, first 3 rank-matched PCs')
    fig.savefig(OUT_DIR / 'robustness_paired_comparison.png', dpi=160)
    plt.show()
    plt.close(fig)
else:
    print('Optional repetitions disabled (N_REPEATS=0).')

## Save provenance and interpret the contrasts

- **Full average → full concatenation:** aggregation effect, with all MEG participants and sources retained.
- **Full average → coverage average:** coverage and feature-multiplicity change at the same participant aggregation.
- **Coverage average → paired coverage:** participant averaging versus a paired pseudo-population, with the same number of electrode slots; participant inclusion also changes if cohort sizes differ.
- **Paired coverage → random control:** anatomical sampling versus spatially random sampling with the same participants, electrode counts and (by default) duplication multiplicities.

The last contrast is the most direct coverage control here. A single pairing is descriptive: inspect optional repeated results before making a robustness claim. High temporal correspondence can arise from common auditory stimulus timing. Weight agreement depends on reference, orientation, source mixing and estimation reliability; it is not a direct measure of identical neural organisation.

In [ ]:
import platform
import scipy
import matplotlib
manifest = {
    'synthetic_demo': SYNTHETIC_DEMO, 'meg_dir': str(MEG_DIR), 'ieeg_dir': str(IEEG_DIR),
    'metadata_csv': str(METADATA_CSV) if METADATA_CSV is not None else None,
    'project_path': str(PROJECT_PATH) if PROJECT_PATH is not None else None,
    'meg_coordinate_unit': MEG_COORDINATE_UNIT, 'ieeg_coordinate_unit': IEEG_COORDINATE_UNIT,
    'meg_times_file': str(MEG_TIMES_FILE) if MEG_TIMES_FILE is not None else None,
    'meg_tmin': MEG_TMIN, 'sfreq': SFREQ, 'conditions': CONDITIONS,
    'condition_mode': CONDITION_MODE, 'meg_scaling': MEG_SCALING,
    'ieeg_scaling': IEEG_SCALING, 'ieeg_multiplier': IEEG_MULTIPLIER,
    'seed': SEED, 'pairing': pairing, 'requested_pairing': PAIRING,
    'random_preserve_duplicates': RANDOM_PRESERVE_DUPLICATES,
    'n_components': N_COMPONENTS, 'n_repeats': N_REPEATS,
    'meg_subjects': inputs['meg_subjects'],
    'python': platform.python_version(), 'numpy': np.__version__,
    'scipy': scipy.__version__, 'pandas': pd.__version__, 'matplotlib': matplotlib.__version__,
}
(OUT_DIR / 'analysis_manifest.json').write_text(json.dumps(manifest, indent=2))
np.save(OUT_DIR / 'times.npy', inputs['times'])
print(f'Saved figures, variance tables, PCA arrays, correlations, matching metadata and provenance to {OUT_DIR}')